<a href="https://colab.research.google.com/github/Agoston03/Machine-Learning-VIMIMA05/blob/main/ML_2026_lab3_Logreg_Autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic Regression and Automatic Differentiation in Pytorch

## **Logistic Regression**

In this part of the session, you will implement a logistic regression model, *i.e.* a linear model for binary classification.

## Data generation

Use the code below to generate some training and test data. The training data consists of two normally distributed blobs, whereas the test data lie on an evenly spaced $100\times 100$ grid, covering the relevant part of the input space (for easy visualization).

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

# Training data with two blobs
x = np.vstack((
    np.vstack((np.random.randn(100)-1,np.random.randn(100)-2)).T,
    np.vstack((np.random.randn(100),np.random.randn(100))).T
))
y = np.concatenate((np.zeros(100),np.ones(100)))

# Test data on a 100x100 grid
gr = np.linspace(-5,5,100)
gx,gy = np.meshgrid(gr,gr)
x_test = np.vstack([gx.flatten(),gy.flatten()]).T

plt.figure(figsize=(10, 10))
plt.scatter(x_test[:,0],x_test[:,1], alpha=0.5, s=3, c='black')
plt.scatter(x[:,0],x[:,1],c=y);

**Exercise 1.** Implement the gradient descent scheme for training a logistic regression model. Recall that it consists of the following steps:

1. Transform the inputs with $\phi: \mathbf x_i \mapsto \left[ 1 \quad \mathbf x_i\right]$. The result should look something like
```
array([[ 1.  ,  2.36, -3.54],
          [ 1.  , -0.96, -0.94],
          [ 1.  , -0.94, -1.35],
          [ 1.  , -0.37, -1.04],
          ...
```
If you want, you can experiment with other transformations to perform nonlinear classification.

2. Initialize the parameter vector $\mathbf w$ *e.g.* to zeros.

3. Compute the gradient
\begin{align*}
\nabla_{\mathbf w}L &=  -\sum_i (y_i - \sigma(\mathbf w^\top \phi(\mathbf x_i))) \phi_i,
\end{align*}
preferably in matrix form,
\begin{align*}
\nabla_{\mathbf w}L &=  \mathbf \Phi^\top \left(\mathbf y - \sigma (\mathbf \Phi \mathbf w)\right) .
\end{align*}

4. Perform the update
\begin{align*}
\mathbf w \gets \mathbf w - \eta \cdot \nabla_{\mathbf w}L,
\end{align*}
where $\eta$ is the learning rate.

5. Repeat Steps 3-4 for a fixed number of steps.

6. Compute predictions as
\begin{align*}
\mathbf y_{\text{test}} = \sigma(\mathbf \Phi_{\text{test}} \mathbf w).
\end{align*}

Plot the results and observe if you managed to get a good fit. You can compare your algorithm to [`sklearn.linear_model.LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

In [ ]:
# Numerically stable implementation of the sigmoid function
def sigmoid(a):
  u = np.exp(-np.abs(a))
  return np.where(a>0,1/(1+u),u/(1+u))

Naturally, we can also find sigmoid implementation as a part of Python packages. For example, [SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.expit.html) implements it; however, we shall always check if an implementation is numerically stable or not.

In [ ]:
# Compute Phi and Phi_test
Phi = ...
Phi_test = ...

# Initialize the parameter vector w
w = ...
η = ... # \eta + ctrl+space or command+space on mac (requires connected colab runtime)

# Perform gradient descent with a suitable learning rate
N_steps = ...
for _ in range(N_steps):
  # calculate gradients and modify w

# Compute predictions
y_pred_manual = ...

In [ ]:
# For comparison
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(x,y)
y_test_sklearn = model.predict(x_test)

In [ ]:
f, ax = plt.subplots(1,2, figsize=(10,5))

# TODO plot your results

# contourf is used the draw the filled regions
ax[1].contourf(gx,gy,y_test_sklearn.reshape(100,100),alpha=0.1)
ax[1].set_xlim(-5, 5)
ax[1].set_ylim(-5, 5)
ax[1].set_title('sklearn prediction')
ax[1].scatter(x[:,0],x[:,1],c=y);


#### **Homework assignment**

So far, we used polynomial kernels. However, it is also possible to use more complex kernels, such as the Radial Basis Function (RBF), aka. Gaussian kernel. In the following, check how this kernel can be used in the implementation of the `scikit-learn` package.

Derive the formula for the gradient of the regularized loss function (see *e.g.,* the lecture notes). Train a regularized nonlinear logistic regression model on the previous data and try to underfit/overfit it:

In [ ]:
# The main difference for this exercise is the way Phi is computed.
# The transformed version of 1 sample, is its RBF value at the same base points.
# e.g. with 3 base points the transformed first sample would be
# [rbf(sample1, base1), rbf(sample1, base2), rbf(sample1, base3)]

from sklearn.gaussian_process.kernels import RBF

# define a grid of base points
base_tmp = np.linspace(-5,5,20)
base = np.dstack(np.meshgrid(base_tmp, base_tmp)).reshape(-1, 2)

# defining a function with an RBF size and a regularization param, for easier reuse of code
def pred_one(γ, λ):
  # γ inverse of the RBF radius (\gamma)
  # λ regularization coeff (\lambda)
  # Construct RBF kernel
  kernel = ...

  # Compute Phi and Phi_test

  # Initialize the parameter vector w

  # Perform gradient descent with a suitable learning rate
  # Use the regularized version

  # Compute predictions
  y_pred = ...
  return y_pred

f,axes = plt.subplots(3,3)
f.set_size_inches(10,10)

for i, axs in enumerate(axes):
  for j, ax in enumerate(axs):
    y_pred_rbf = pred_one(γ=0.3 + i * 1.5, λ=j * 0.005)
    ax.contourf(gx,gy,y_pred_rbf.reshape(100,100),alpha=0.5)
    ax.set_xlim(-5,5)
    ax.set_ylim(-5,5)
    ax.scatter(x[:,0],x[:,1],c=y)

## **Automatic Differentiation in `pytorch`**

For now, we will have a short introduction to automatic differentiation in `pytorch`. For a more in depth guide on how it all works I recommend reading the [official documentation](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).

Here's a short description of the main concepts this will cover:
- [`torch.Tensor.grad`](https://pytorch.org/docs/stable/generated/torch.Tensor.grad.html#torch.Tensor.grad) : this attribute will store the gradient of the tensor, if the tensor has no gradient, it has value [None](https://docs.python.org/3/c-api/none.html).
- [`torch.Tensor.requires_grad`](https://pytorch.org/docs/stable/generated/torch.Tensor.requires_grad.html#torch.Tensor.requires_grad) : A boolean (True or False) attribute. When automatically differentiating pytorch only saves the gradient on tensors with `requires_grad=True`.
- [`torch.Tensor.grad_fn`](https://pytorch.org/tutorials/beginner/former_torchies/autograd_tutorial.html) : Pytorch uses the "chain rule" for derivation, storing the last operation executed on the tensor. Don't worry if this is hard to conceptualize, the example should help to clarify
- **The pytorch dynamic computational graph** : By always storing the operations in `grad_fn` pytorch dynamically builds a computational graph.
- [`torch.Tensor.backward()`](https://pytorch.org/docs/stable/generated/torch.Tensor.backward.html#torch.Tensor.backward) : This function actually calculates the gradients based on the computational graph. It deletes the computational graph from memory, and fills in the the `grad` attributes of the tensors that have `requires_grad=True`.
- [`torch.no_grad()`](https://pytorch.org/docs/stable/generated/torch.no_grad.html) : temporarily disable the gradient calculations using python's [with](https://docs.python.org/3/reference/compound_stmts.html#with) syntax.

To help visualize the computational graph, we will install and use the [torchviz](https://github.com/szagoruyko/pytorchviz) package.

In [ ]:
!pip install torchviz  # torchviz isn't installed in colab by default.
# by starting the line with '!'  we can run commands on the machine
# as if this cell was a command line
# this command uses the default python package manager 'pip'
# to install the package 'torchviz'

In [ ]:
import torch
import torchviz

### Some Examples of Autograd

A simple example: calculate the gradient of $f(x) = 3x^2$ at $x=2$.

Doing this manually would involve calculating the derivative of $3x^2$ with respect to $x$,

$$f(x)' = (3x^2)' = 3(x^2)' = 6x$$

and substituting in $x=2$

$$f(2)' = 6 \cdot 2 = 12$$


In [ ]:
# Make a single-value tensor, with requires_grad=True.
x = torch.tensor(2., requires_grad=True)
# There is no gradient on this tensor yet
x, x.grad

In [ ]:
# doing any operation on this tensor sets grad_fn, with the appropriate operation
x**2 # grad_fn represents squaring operation we just did

In [ ]:
# we can visualize the resulting computation graph using torchviz
# the green box represents x**2, the light blue box is x.
# AccumulateGrad is just telling pytorch to record the
# gradient in x, and PowBackward0 represents the squaring
torchviz.make_dot(x**2)

In [ ]:
# now with the multiplication
3*x**2

In [ ]:
# we can even look into how grad_fn keeps track of the entire graph
(3*x**2).grad_fn.next_functions # we can see our PowBackward0
# is the next

In [ ]:
# visualized, note that the graph changed dynamically, now also
# representing the multiplication with MulBackward0
torchviz.make_dot(3*x**2)

In [ ]:
# We can trigger calculating the gradients by calling .backward()
# backward() can only be called on scalar tensors (tensors holding a single value
(3*x**2).backward()
# we see that the gradient of x is now set, to the expected value of 12
x.grad

In [ ]:
# we can use torch.no_grad() to tell pytorch to stop recording the operations
with torch.no_grad():
  fn = 3*x**2 # code in the with statement has no grad_fn added
fn, fn.grad_fn # even though x requires gradients, grad_fn is None.

To demonstrate a more involved example using a linear layer, let's consider the setup where you have:
- A weight matrix $W \in \mathbb{R}^{M \times N}$
- A bias vector $b \in \mathbb{R}^M$
- An input vector $x \in \mathbb{R}^N$

The linear transformation applied to $x$ using weights $W$ and bias $b$ can be represented as $y = Wx + b$. This operation is common in neural networks, where it is often referred to as a linear layer or a fully connected layer.

For the sake of simplicity, let's just say we want this to always output the zero vector. We could define a loss function of $(y-0)^2$

In [ ]:
# Define the dimensions
N = 5
M = 3

# Create tensors for weights and bias
# Set requires_grad=True to track computations on these tensors for gradient computation
W = torch.randn(M, N, requires_grad=True)
b = torch.randn(M, requires_grad=True)

# Input vector
x = torch.randn(N)

W, b, x

In [ ]:
# Perform the linear transformation
y = torch.matmul(W, x) + b  # using torch.matmul for matrix multiplication
# or y = W @ x + b , but torch.matmul is preferred for clarity and readability
y # we can see that grad_fn was set accordingly

In [ ]:
# we can visualize this computational graph as well
torchviz.make_dot(y)

In [ ]:
loss = (y**2).sum() # calculate the scalar product y*y, which is the loss defined above
loss.backward() # calculate the gradient of the loss with respect to W and b
# this fills in the .grad attributes of W and b
W.grad, b.grad

In [ ]:
# perform gradient descent with a small learning rate
with torch.no_grad(): # we need to disable gradient tracking when modifying our parameters
  W -= 0.03*W.grad
  b -= 0.03*b.grad

In [ ]:
y_after = torch.matmul(W, x) + b
y, y_after # note how the output is closer to 0

### Autograd Dangers!

Automatic differentiation works well in most cases. However, if we want to implement something new for neural networks in the future, we have to respect some rules:

- **Rule1:** While `numpy` (or self-written) packages cannot handle gradients, `pytorch` functions compute them.
- **Rule2:** There still exist functions that are not differentiable!

In [ ]:
#Example for Rule1 (it _shall_ lead to an error, as it can be checked in runtime)
import numpy as np

x = torch.tensor([torch.pi/2], requires_grad=True)
y = np.sin(x) + torch.tensor([1.2])
y.backward()
x.grad

In [ ]:
#TODO: Correct the above code, so that it can run without error, and provides
#the necessary gradients:
...

**Question:**

The above function (`x.grad`), shall be: $\sin(x)' = \cos\big(\frac{\pi}{2}\big) = 0.0$

The above result is not zero. What causes the difference?

### Non-differentiable functions

So far, we saw examples for **rule1**. However, **rule2** might be even more tricky. In this case, we will not get an exception; hence, our code will work (or at least do something), but the result will not exactly be that we expect: we will get a gradient tensor; however, it will be zero or contain some invalid value.

This is a problem if we try to code something new and clever for our neural network architecture in the future, such as a new loss function. So, whenever you implement your own loss function, always check that the resulting gradients are those that you would except.

Run the following cells, and check the results:

In [ ]:
#we define a new method which includes a non-differentiable function:

def my_updated_mse(y_pred, y_true):
  loss = (y_pred - y_true)**2
  loss = torch.clamp(loss, -1, 1) #pretends to be okay...
  return loss

Let us check how it works:

In [ ]:
y_true = torch.tensor([2.0])
y_pred = torch.tensor([4.0], requires_grad=True)
loss = my_updated_mse(y_pred, y_true)
loss.backward()
y_pred.grad

**Question:**

What have you found? In general, how can you avoid losing the gradients in custom loss functions?